# 03 Travel times

Travel times from every station to every cell of the velocity model (eikonal, fast marching), plus 1D travel-time
tables for locating and focusing with a simpler model. Slow the first time; existing tables are skipped on reruns.

Inputs: `data/velocity_model.npz`, `data/stations.csv`. Output: `data/tt/`

In [1]:
# --- Imports ---
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import skfmm

In [3]:
# --- Settings ---
PHASES = ("p",)
CROP = 13.0 
R0 = 0.3 # radius of the starting sphere around each station (km)
TT_DIR = "../data/tt"

# --- Inputs ---
m = np.load("../data/velocity_model.npz")
x, y, z = m["x"], m["y"], m["z"]
vel = {"p": m["vp"], "s": m["vs"]}
stations = pd.read_csv("../data/stations.csv", index_col="station")

dx = [x[1] - x[0], y[1] - y[0], z[1] - z[0]]
XX, YY, ZZ = np.meshgrid(x, y, z, indexing="ij")
ix, iy = np.abs(x) <= CROP, np.abs(y) <= CROP

In [4]:
def travel_time(sx, sy, v):
    d = np.sqrt((XX - sx) ** 2 + (YY - sy) ** 2 + ZZ ** 2)               # straight-line distance
    v0 = v[np.argmin(abs(x - sx)), np.argmin(abs(y - sy)), 0]            # velocity at the station
    t = skfmm.travel_time(d - R0, v, dx=dx, order=2)                     # wavefront starts on the sphere
    t = np.where(d < R0, d / v0, t + R0 / v0)
    return t[ix][:, iy].astype(np.float16)

In [ ]:
# --- One table per station and phase ---
os.makedirs(TT_DIR, exist_ok=True)
np.savez(f"{TT_DIR}/grid.npz", x=x[ix], y=y[iy], z=z)

for ph in PHASES:
    for s, row in stations.iterrows():
        path = f"{TT_DIR}/{s:03d}_{ph}.npy"
        if not os.path.exists(path):
            np.save(path, travel_time(row.x, row.y, vel[ph]))
print(len([f for f in os.listdir(TT_DIR) if f.endswith(".npy")]), "tables")

In [ ]:
# --- 1D model: the 3D model averaged at each depth ---
r1 = np.arange(0, 30.01, 0.1)
d1 = np.hypot(*np.meshgrid(r1, z, indexing="ij"))

tt1d = {}
for ph in PHASES:
    v1 = vel[ph].mean(axis=(0, 1))
    v2 = np.tile(v1, (len(r1), 1))
    t = skfmm.travel_time(d1 - R0, v2, dx=[r1[1] - r1[0], dx[2]], order=2)
    tt1d[ph] = np.where(d1 < R0, d1 / v1[0], t + R0 / v1[0])
np.savez(f"{TT_DIR}/tt1d.npz", r=r1, z=z, **{f"t_{ph}": t for ph, t in tt1d.items()})

In [ ]:
# --- Travel-time contours on a section through one station ---
STATION = 0
AZ = 120 

g = np.load(f"{TT_DIR}/grid.npz")
T = np.load(f"{TT_DIR}/{STATION:03d}_p.npy").astype(np.float32)
sx, sy = stations.loc[STATION, ["x", "y"]]
s = np.linspace(-10, 10, 201)
jx = np.clip(np.round((sx + s * np.sin(np.radians(AZ)) - g["x"][0]) / dx[0]).astype(int), 0, len(g["x"]) - 1)
jy = np.clip(np.round((sy + s * np.cos(np.radians(AZ)) - g["y"][0]) / dx[1]).astype(int), 0, len(g["y"]) - 1)

with plt.style.context("default"):
    fig, ax = plt.subplots(figsize=(9, 4))
    vsec = vel["p"][np.searchsorted(x, g["x"][jx]), np.searchsorted(y, g["y"][jy]), :]
    pc = ax.pcolormesh(s, z, vsec.T, cmap="viridis", shading="auto")
    cs = ax.contour(s, z, T[jx, jy, :].T, levels=np.arange(0.25, 4, 0.25), colors="w", linewidths=0.8)
    ax.clabel(cs, fmt="%.2f s", fontsize=7)
    ax.plot(0, 0, "rv", ms=8, clip_on=False)
    ax.set(ylim=(8, 0), xlabel="Distance from station along section (km)", ylabel="Depth (km)",
           title=f"P travel times from station {STATION}")
    fig.colorbar(pc, label="Vp (km/s)")
    plt.show()

In [ ]:
# --- 3D minus 1D travel time, straight below and at distance ---
from scipy.interpolate import RectBivariateSpline
t1 = RectBivariateSpline(r1, z, tt1d["p"])
for depth in (2, 4, 6, 8):
    k = np.argmin(abs(g["z"] - depth))
    rr = np.hypot(g["x"][:, None] - sx, g["y"][None, :] - sy)
    diff = T[:, :, k] - t1.ev(rr, depth)
    print(f"{depth} km: 3D - 1D = {diff.mean() * 1000:+.0f} ms on average, "
          f"range {diff.min() * 1000:+.0f} to {diff.max() * 1000:+.0f} ms")